### Cell 07.01 — define the frozen LRN QTL region

In [ ]:
# Cell 07.01
# Frozen LRN suggestive QTL summary from Notebook 04/05.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


LRN_QTL = {
    "trait": "lrn",
    "peak_marker": "Satt282a",
    "structural_group": "pLG04",
    "physical_chr": "Gm02",
    "physical_assignment": "strong",

    "lod": 2.207883,
    "empirical_p": 0.052895,
    "significance_status": "suggestive_10pct",

    "region_start_bp": 20_165_739,
    "region_end_bp": 46_904_614,

    "region_definition":
        "anchor_defined_physical_bracket",

    "peak_direct_anchor_bp": 30_354_248,
    "peak_anchor_evidence":
        "assay_family"
}


LRN_QTL[
    "region_span_mb"
] = (
    LRN_QTL["region_end_bp"]
    - LRN_QTL["region_start_bp"]
) / 1e6


print("FROZEN LRN QTL")
print("=" * 90)

for key, value in LRN_QTL.items():
    print(f"{key}: {value}")

### Cell 07.02 — load the existing Wm82.gnm6 gene annotation
* Since you already downloaded and parsed the same annotation in Notebook 06, we can reuse it directly.

In [ ]:
# Cell 07.02
# Load Wm82.gnm6 gene annotation.

import gzip


SOYBASE_REF_DIR = (
    PROJECT_ROOT
    / "reference"
    / "soybase"
)


GNM6_GENE_GFF = (
    SOYBASE_REF_DIR
    / "glyma.Wm82.gnm6.ann1.PKSW.gene_models_main.gff3.gz"
)


print(
    "GNM6 ANNOTATION EXISTS:",
    GNM6_GENE_GFF.exists()
)


gff_columns = [
    "seqid",
    "source",
    "feature_type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes"
]


gnm6_gff = pd.read_csv(
    GNM6_GENE_GFF,
    sep="\t",
    comment="#",
    names=gff_columns,
    dtype={
        "seqid": str,
        "source": str,
        "feature_type": str,
        "score": str,
        "strand": str,
        "phase": str,
        "attributes": str
    }
)


gnm6_gff[
    "start"
] = pd.to_numeric(
    gnm6_gff["start"],
    errors="coerce"
)

gnm6_gff[
    "end"
] = pd.to_numeric(
    gnm6_gff["end"],
    errors="coerce"
)


gnm6_genes = (
    gnm6_gff
    .loc[
        gnm6_gff[
            "feature_type"
        ] == "gene"
    ]
    .copy()
)


print(
    "TOTAL GENE FEATURES:",
    len(gnm6_genes)
)

### Cell 07.03 — parse gene attributes

In [ ]:
# Cell 07.03
# Parse GFF3 attributes for gene-level annotation.

def parse_gff_attributes(text):

    result = {}

    if pd.isna(text):
        return result

    for item in str(text).split(";"):

        if "=" in item:

            key, value = item.split(
                "=",
                1
            )

            result[key] = value

    return result


gnm6_genes[
    "attribute_dict"
] = (
    gnm6_genes[
        "attributes"
    ]
    .map(
        parse_gff_attributes
    )
)


for field in [
    "ID",
    "Name",
    "Note",
    "Ontology_term",
    "Dbxref",
    "ancestorIdentifier"
]:

    gnm6_genes[field] = (
        gnm6_genes[
            "attribute_dict"
        ]
        .map(
            lambda d, f=field:
                d.get(
                    f,
                    np.nan
                )
        )
    )


print(
    "GENES WITH NAME:",
    gnm6_genes[
        "Name"
    ].notna().sum()
)

print(
    "GENES WITH NOTE:",
    gnm6_genes[
        "Note"
    ].notna().sum()
)

### Cell 07.04 — extract all genes in the LRN Gm02 bracket

In [ ]:
# Cell 07.04
# Extract all Wm82.gnm6 genes overlapping the
# LRN anchor-defined Gm02 physical bracket.

LRN_SEQID = (
    "glyma.Wm82.gnm6.Gm02"
)

LRN_START_BP = (
    LRN_QTL[
        "region_start_bp"
    ]
)

LRN_END_BP = (
    LRN_QTL[
        "region_end_bp"
    ]
)


lrn_gm02_genes = (
    gnm6_genes
    .loc[
        (
            gnm6_genes[
                "seqid"
            ]
            == LRN_SEQID
        )
        &
        (
            gnm6_genes[
                "end"
            ]
            >= LRN_START_BP
        )
        &
        (
            gnm6_genes[
                "start"
            ]
            <= LRN_END_BP
        )
    ]
    .copy()
    .sort_values(
        [
            "start",
            "end"
        ]
    )
    .reset_index(drop=True)
)


lrn_gm02_genes[
    "start_mb"
] = (
    lrn_gm02_genes[
        "start"
    ] / 1e6
)

lrn_gm02_genes[
    "end_mb"
] = (
    lrn_gm02_genes[
        "end"
    ] / 1e6
)


print(
    "LRN Gm02 GENE INVENTORY"
)

print("=" * 90)

print(
    f"Region: "
    f"{LRN_START_BP:,}–"
    f"{LRN_END_BP:,} bp"
)

print(
    f"Span: "
    f"{LRN_QTL['region_span_mb']:.3f} Mb"
)

print(
    "Genes overlapping region:",
    len(lrn_gm02_genes)
)


display(
    lrn_gm02_genes[
        [
            "Name",
            "start",
            "end",
            "start_mb",
            "end_mb",
            "strand",
            "Note",
            "Ontology_term"
        ]
    ]
    .head(20)
)

### Cell 07.05 — decode annotations and build a root-development keyword screen

In [ ]:
# Cell 07.05
# Decode GFF annotation text and identify genes with
# root-development / hormone / meristem / cell-wall relevance.

from urllib.parse import unquote


lrn_gm02_genes[
    "Note_decoded"
] = (
    lrn_gm02_genes[
        "Note"
    ]
    .fillna("")
    .map(unquote)
)


root_candidate_rules = {

    "auxin_signaling": [
        "auxin",
        "arf",
        "aux/iaa",
        "transport inhibitor response",
        "tir1",
        "pin-formed",
        "pin protein",
        "auxin efflux",
        "auxin influx"
    ],

    "cytokinin_signaling": [
        "cytokinin",
        "response regulator",
        "arr",
        "histidine kinase"
    ],

    "gibberellin_signaling": [
        "gibberellin",
        "gibberellic",
        "della"
    ],

    "root_meristem_development": [
        "root meristem",
        "root development",
        "lateral root",
        "root hair",
        "root cap",
        "meristem",
        "root growth"
    ],

    "transcription_regulation": [
        "transcription factor",
        "bhlh",
        "myb",
        "nac",
        "wrky",
        "homeobox",
        "lbd",
        "lateral organ boundaries"
    ],

    "cell_wall_remodeling": [
        "expansin",
        "pectate lyase",
        "pectin",
        "cell wall",
        "cellulose",
        "xyloglucan"
    ],

    "calcium_signaling": [
        "calmodulin",
        "calcium-dependent",
        "calcium/calmodulin",
        "calcium binding",
        "calcium-binding"
    ],

    "phosphoinositide_signaling": [
        "phosphatidylinositol",
        "phosphoinositide",
        "pip kinase"
    ]
}


def classify_root_candidate(note):

    text = str(note).lower()

    hits = []

    for category, terms in (
        root_candidate_rules.items()
    ):

        if any(
            term in text
            for term in terms
        ):
            hits.append(category)

    return ";".join(hits)


lrn_gm02_genes[
    "root_candidate_categories"
] = (
    lrn_gm02_genes[
        "Note_decoded"
    ]
    .map(
        classify_root_candidate
    )
)


lrn_root_screen = (
    lrn_gm02_genes
    .loc[
        lrn_gm02_genes[
            "root_candidate_categories"
        ] != ""
    ]
    .copy()
    .sort_values(
        ["start", "Name"]
    )
    .reset_index(drop=True)
)


print(
    "ROOT-DEVELOPMENT SCREEN:",
    len(lrn_root_screen),
    "of",
    len(lrn_gm02_genes),
    "genes"
)


display(
    lrn_root_screen[
        [
            "Name",
            "start_mb",
            "end_mb",
            "strand",
            "root_candidate_categories",
            "Note_decoded"
        ]
    ]
)

### Cell 07.06 — make a stricter primary-annotation screen
* As with the SCN analysis, this avoids false positives caused only by InterPro/domain text.

In [ ]:
# Cell 07.06
# Use the primary annotation title before the first semicolon.

def primary_annotation_title(note):

    if pd.isna(note):
        return ""

    return (
        str(note)
        .split(";")[0]
        .strip()
    )


lrn_gm02_genes[
    "primary_annotation"
] = (
    lrn_gm02_genes[
        "Note_decoded"
    ]
    .map(
        primary_annotation_title
    )
)


def classify_primary_root_annotation(text):

    text = str(text).lower()

    hits = []

    for category, terms in (
        root_candidate_rules.items()
    ):

        if any(
            term in text
            for term in terms
        ):
            hits.append(category)

    return ";".join(hits)


lrn_gm02_genes[
    "title_based_root_categories"
] = (
    lrn_gm02_genes[
        "primary_annotation"
    ]
    .map(
        classify_primary_root_annotation
    )
)


lrn_title_screen = (
    lrn_gm02_genes
    .loc[
        lrn_gm02_genes[
            "title_based_root_categories"
        ] != ""
    ]
    .copy()
    .sort_values(
        ["start", "Name"]
    )
    .reset_index(drop=True)
)


print(
    "TITLE-BASED ROOT SCREEN:",
    len(lrn_title_screen),
    "of",
    len(lrn_gm02_genes),
    "genes"
)


display(
    lrn_title_screen[
        [
            "Name",
            "start_mb",
            "end_mb",
            "strand",
            "title_based_root_categories",
            "primary_annotation"
        ]
    ]
)

### Cell 07.07 — prioritize biologically stronger root-development classes
* For lateral-root number, I would give more weight to auxin/cytokinin, root-development annotations, and hormone-signaling regulators than to generic transcription factors.

In [ ]:
# Cell 07.07
# Assign transparent functional priority for the LRN phenotype.

def assign_lrn_priority(row):

    cats = str(
        row[
            "title_based_root_categories"
        ]
    )

    title = str(
        row[
            "primary_annotation"
        ]
    ).lower()


    # Tier 1:
    # strongest direct relevance to lateral/root development
    # or major root-development hormone signaling.

    if (
        "root_meristem_development"
        in cats
        or
        "auxin_signaling"
        in cats
        or
        "cytokinin_signaling"
        in cats
    ):
        return "Tier_1"


    # Tier 2:
    # developmental hormone/signaling systems
    # and cell-wall remodeling relevant to root emergence.

    if (
        "gibberellin_signaling"
        in cats
        or
        "calcium_signaling"
        in cats
        or
        "phosphoinositide_signaling"
        in cats
        or
        "cell_wall_remodeling"
        in cats
    ):
        return "Tier_2"


    # Tier 3:
    # broader transcriptional regulators.

    if (
        "transcription_regulation"
        in cats
    ):
        return "Tier_3"


    return "Tier_3"


lrn_title_screen[
    "priority_tier"
] = (
    lrn_title_screen
    .apply(
        assign_lrn_priority,
        axis=1
    )
)


tier_rank = {
    "Tier_1": 1,
    "Tier_2": 2,
    "Tier_3": 3
}


lrn_title_screen[
    "priority_rank"
] = (
    lrn_title_screen[
        "priority_tier"
    ]
    .map(tier_rank)
)


lrn_priority = (
    lrn_title_screen
    .sort_values(
        [
            "priority_rank",
            "start_mb"
        ]
    )
    .reset_index(drop=True)
)


display(
    lrn_priority[
        [
            "Name",
            "start_mb",
            "priority_tier",
            "title_based_root_categories",
            "primary_annotation"
        ]
    ]
)

### Cell 07.08 — measure proximity to the Satt282a physical anchor
* Because Satt282a itself has a family-based physical position near 30.354 Mb, proximity is useful as a descriptive feature, but not enough to define a narrower QTL interval.

In [ ]:
# Cell 07.08
# Add descriptive distance from the Satt282a family-based anchor.
# This does NOT define a statistical confidence interval.

LRN_PEAK_ANCHOR_MB = (
    LRN_QTL[
        "peak_direct_anchor_bp"
    ] / 1e6
)


lrn_priority[
    "midpoint_mb"
] = (
    lrn_priority[
        "start_mb"
    ]
    + lrn_priority[
        "end_mb"
    ]
) / 2


lrn_priority[
    "distance_to_Satt282a_anchor_mb"
] = (
    lrn_priority[
        "midpoint_mb"
    ]
    - LRN_PEAK_ANCHOR_MB
).abs()


lrn_priority = (
    lrn_priority
    .sort_values(
        [
            "priority_rank",
            "distance_to_Satt282a_anchor_mb"
        ]
    )
    .reset_index(drop=True)
)


print(
    "Satt282a family-based physical anchor:",
    f"{LRN_PEAK_ANCHOR_MB:.6f} Mb"
)


display(
    lrn_priority[
        [
            "Name",
            "start_mb",
            "priority_tier",
            "title_based_root_categories",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation"
        ]
    ]
)

### Cell 07.09 — conservative mechanism classification

In [ ]:
# Cell 07.09
# Conservative classification based on explicit primary annotations.
# Avoid broad substring matches such as "ARR" or annotation-domain text.

def conservative_lrn_class(title):

    t = str(title).lower()

    # Auxin
    if (
        "auxin response factor" in t
        or "auxin-responsive" in t
        or "auxin responsive" in t
        or "transport inhibitor response 1" in t
        or "tir1" in t
        or "aux/iaa" in t
        or "auxin efflux" in t
        or "auxin influx" in t
    ):
        return "auxin"

    # Cytokinin
    if (
        "cytokinin" in t
        or "two-component response regulator" in t
    ):
        return "cytokinin"

    # Gibberellin
    if (
        "gibberellin" in t
        or "della" in t
    ):
        return "gibberellin"

    # Calcium signaling
    if (
        "calcium-dependent protein kinase" in t
        or "calcium/calmodulin-dependent" in t
        or "calmodulin-binding" in t
        or "calcium-dependent lipid-binding" in t
    ):
        return "calcium_signaling"

    # Phosphoinositide signaling
    if (
        "phosphatidylinositol" in t
        or "phosphoinositide" in t
    ):
        return "phosphoinositide_signaling"

    # Cell wall
    if (
        "cellulose synthase" in t
        or "expansin" in t
        or "pectate lyase" in t
        or "xyloglucan" in t
    ):
        return "cell_wall_remodeling"

    return ""


lrn_gm02_genes[
    "conservative_lrn_class"
] = (
    lrn_gm02_genes[
        "primary_annotation"
    ]
    .map(conservative_lrn_class)
)


lrn_mechanistic_candidates = (
    lrn_gm02_genes
    .loc[
        lrn_gm02_genes[
            "conservative_lrn_class"
        ] != ""
    ]
    .copy()
)


lrn_mechanistic_candidates[
    "midpoint_mb"
] = (
    lrn_mechanistic_candidates["start_mb"]
    + lrn_mechanistic_candidates["end_mb"]
) / 2


lrn_mechanistic_candidates[
    "distance_to_Satt282a_anchor_mb"
] = (
    lrn_mechanistic_candidates["midpoint_mb"]
    - LRN_PEAK_ANCHOR_MB
).abs()


lrn_mechanistic_candidates = (
    lrn_mechanistic_candidates
    .sort_values(
        [
            "conservative_lrn_class",
            "distance_to_Satt282a_anchor_mb"
        ]
    )
    .reset_index(drop=True)
)


print(
    "CONSERVATIVE MECHANISTIC CANDIDATES:",
    len(lrn_mechanistic_candidates)
)


display(
    lrn_mechanistic_candidates[
        [
            "Name",
            "start_mb",
            "conservative_lrn_class",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation"
        ]
    ]
)

### Cell 07.10 — specifically extract the hormone candidates

In [ ]:
# Cell 07.10
# Focus on auxin/cytokinin/gibberellin candidates,
# which have the clearest mechanistic connection
# to lateral-root development.

hormone_classes = [
    "auxin",
    "cytokinin",
    "gibberellin"
]


lrn_hormone_candidates = (
    lrn_mechanistic_candidates
    .loc[
        lrn_mechanistic_candidates[
            "conservative_lrn_class"
        ].isin(hormone_classes)
    ]
    .copy()
    .sort_values(
        "distance_to_Satt282a_anchor_mb"
    )
    .reset_index(drop=True)
)


print(
    "HORMONE-RELATED CANDIDATES:",
    len(lrn_hormone_candidates)
)


display(
    lrn_hormone_candidates[
        [
            "Name",
            "start_mb",
            "end_mb",
            "conservative_lrn_class",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation"
        ]
    ]
)

### Cell 07.11 — inspect genes near the Satt282a physical anchor without keyword bias
* This is important because annotation screening could miss an unexpected but biologically important gene.

In [ ]:
# Cell 07.11
# Independently inspect all genes within ±1 Mb of the
# family-based Satt282a physical anchor.
#
# This window is descriptive only and is NOT a QTL interval.

LOCAL_WINDOW_MB = 1.0


lrn_near_peak_anchor = (
    lrn_gm02_genes
    .loc[
        (
            lrn_gm02_genes["end_mb"]
            >= LRN_PEAK_ANCHOR_MB - LOCAL_WINDOW_MB
        )
        &
        (
            lrn_gm02_genes["start_mb"]
            <= LRN_PEAK_ANCHOR_MB + LOCAL_WINDOW_MB
        )
    ]
    .copy()
)


lrn_near_peak_anchor[
    "midpoint_mb"
] = (
    lrn_near_peak_anchor["start_mb"]
    + lrn_near_peak_anchor["end_mb"]
) / 2


lrn_near_peak_anchor[
    "distance_to_Satt282a_anchor_mb"
] = (
    lrn_near_peak_anchor["midpoint_mb"]
    - LRN_PEAK_ANCHOR_MB
).abs()


lrn_near_peak_anchor = (
    lrn_near_peak_anchor
    .sort_values(
        "distance_to_Satt282a_anchor_mb"
    )
    .reset_index(drop=True)
)


print(
    "GENES WITHIN ±1 Mb OF Satt282a ANCHOR:",
    len(lrn_near_peak_anchor)
)


display(
    lrn_near_peak_anchor[
        [
            "Name",
            "start_mb",
            "end_mb",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation"
        ]
    ]
)

### Cell 07.12 — save this cleaned checkpoint

In [ ]:
# Cell 07.12
# Save the cleaned LRN candidate-region checkpoint.

LRN_CANDIDATE_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_lrn_gm02_candidate_screen.xlsx"
)


with pd.ExcelWriter(
    LRN_CANDIDATE_FILE,
    engine="openpyxl"
) as writer:

    lrn_gm02_genes.to_excel(
        writer,
        sheet_name="all_429_genes",
        index=False
    )

    lrn_mechanistic_candidates.to_excel(
        writer,
        sheet_name="mechanistic_candidates",
        index=False
    )

    lrn_hormone_candidates.to_excel(
        writer,
        sheet_name="hormone_candidates",
        index=False
    )

    lrn_near_peak_anchor.to_excel(
        writer,
        sheet_name="near_Satt282a_1Mb",
        index=False
    )


print("Saved:")
print(LRN_CANDIDATE_FILE)

### Cell 07.13 — flag candidates requiring annotation reconciliation

In [ ]:
# Cell 07.13
# Record cases where external literature/older assemblies
# give annotations that differ from the current Wm82.gnm6 GFF.

annotation_reconciliation = pd.DataFrame(
    [
        {
            "gene_gnm6": "Glyma.02G211800",
            "gnm6_annotation":
                "transport inhibitor response 1-like protein-like",
            "older_literature_annotation":
                "AFB4/5-family auxin F-box receptor",
            "functional_consistency":
                "consistent_auxin_receptor_family",
            "needs_gene_correspondence_check":
                True,
        },
        {
            "gene_gnm6": "Glyma.02G162600",
            "gnm6_annotation":
                "two-component response regulator ARR2-like",
            "older_literature_annotation":
                "LUX circadian-clock transcription factor",
            "functional_consistency":
                "discordant",
            "needs_gene_correspondence_check":
                True,
        },
        {
            "gene_gnm6": "Glyma.02G200500",
            "gnm6_annotation":
                "auxin response factor 18-like",
            "older_literature_annotation":
                "B3/ABI3-like transcription-factor annotation reported",
            "functional_consistency":
                "discordant_or_uncertain",
            "needs_gene_correspondence_check":
                True,
        },
    ]
)


display(annotation_reconciliation)

### Cell 07.14 — build a conservative literature-ready shortlist
* For now, keep candidates that are either mechanistically strong or unusually close to the marker anchor, but do not yet assign direct literature evidence to the ambiguous IDs.

In [ ]:
# Cell 07.14
# Literature-ready shortlist.
# This is intentionally small and transparent.

priority_gene_names = [
    # Auxin / cytokinin / GA
    "Glyma.02G211800",
    "Glyma.02G200500",
    "Glyma.02G202200",
    "Glyma.02G198300",
    "Glyma.02G164148",

    # Strong signaling near Satt282a
    "Glyma.02G165433",
    "Glyma.02G163800",

    # Developmental transcription factor near the anchor
    "Glyma.02G166400",
]


lrn_literature_shortlist = (
    lrn_gm02_genes
    .loc[
        lrn_gm02_genes[
            "Name"
        ].isin(priority_gene_names)
    ]
    .copy()
)


lrn_literature_shortlist[
    "midpoint_mb"
] = (
    lrn_literature_shortlist["start_mb"]
    + lrn_literature_shortlist["end_mb"]
) / 2


lrn_literature_shortlist[
    "distance_to_Satt282a_anchor_mb"
] = (
    lrn_literature_shortlist["midpoint_mb"]
    - LRN_PEAK_ANCHOR_MB
).abs()


lrn_literature_shortlist = (
    lrn_literature_shortlist
    .sort_values(
        "distance_to_Satt282a_anchor_mb"
    )
    .reset_index(drop=True)
)


display(
    lrn_literature_shortlist[
        [
            "Name",
            "start_mb",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation",
            "ancestorIdentifier"
        ]
    ]
)

### Cell 07.15 — inspect assembly identifiers for the shortlist

In [ ]:
# Cell 07.15
# Determine what historical gene-model identifiers are
# already available directly from the gnm6 annotation.

assembly_crosswalk_candidates = (
    lrn_literature_shortlist[
        [
            "Name",
            "start",
            "end",
            "start_mb",
            "primary_annotation",
            "ancestorIdentifier",
            "Dbxref"
        ]
    ]
    .copy()
)


print("ASSEMBLY-AWARE CANDIDATE TABLE")
print("=" * 90)

display(assembly_crosswalk_candidates)

### Cell 07.16 — add only the literature evidence that is currently defensible

In [ ]:
# Cell 07.16
# Add literature evidence conservatively.
#
# "direct_root_trait_evidence" means evidence involving
# root architecture/root development in soybean.
# Generic auxin-family membership is kept separate.

lrn_literature_shortlist[
    "soybean_functional_literature"
] = False

lrn_literature_shortlist[
    "direct_root_trait_evidence"
] = False

lrn_literature_shortlist[
    "literature_evidence_type"
] = ""

lrn_literature_shortlist[
    "literature_note"
] = ""


# Glyma.02G211800:
# independently classified in the soybean TIR1/AFB auxin receptor family.
mask = (
    lrn_literature_shortlist["Name"]
    == "Glyma.02G211800"
)

lrn_literature_shortlist.loc[
    mask,
    "soybean_functional_literature"
] = True

lrn_literature_shortlist.loc[
    mask,
    "literature_evidence_type"
] = "soybean_auxin_receptor_family"

lrn_literature_shortlist.loc[
    mask,
    "literature_note"
] = (
    "Reported as GmAFB4/5_D.1 in the soybean "
    "TIR1/AFB auxin-response gene family. "
    "Supports auxin-signaling plausibility; "
    "not direct evidence for lateral-root-number variation."
)


display(
    lrn_literature_shortlist[
        [
            "Name",
            "start_mb",
            "primary_annotation",
            "soybean_functional_literature",
            "direct_root_trait_evidence",
            "literature_evidence_type",
            "literature_note"
        ]
    ]
)

### Cell 07.17 — record literature and assembly evidence explicitly

In [ ]:
# Cell 07.17
# Evidence table for the eight literature-ready LRN candidates.
#
# Evidence categories are deliberately conservative:
#   - exact soybean evidence
#   - family/mechanistic plausibility
#   - unresolved annotation/cross-assembly issue
#
# None of these categories implies causal control of LRN.

lrn_evidence = lrn_literature_shortlist.copy()


lrn_evidence[
    "exact_soybean_gene_evidence"
] = False

lrn_evidence[
    "soybean_development_trait_evidence"
] = False

lrn_evidence[
    "annotation_consistency"
] = "not_independently_checked"

lrn_evidence[
    "evidence_note"
] = ""


# ---------------------------------------------------------
# Glyma.02G211800
# Strongest independently supported candidate.
# ---------------------------------------------------------

mask = (
    lrn_evidence["Name"]
    == "Glyma.02G211800"
)

lrn_evidence.loc[
    mask,
    "exact_soybean_gene_evidence"
] = True

lrn_evidence.loc[
    mask,
    "soybean_development_trait_evidence"
] = True

lrn_evidence.loc[
    mask,
    "annotation_consistency"
] = "consistent_auxin_receptor_family"

lrn_evidence.loc[
    mask,
    "evidence_note"
] = (
    "Independently identified as GmAFB4/5_D.1, "
    "an AFB/TIR1-family auxin receptor. Also proposed "
    "as an auxin-pathway candidate in soybean first-pod-height "
    "mapping. No direct lateral-root-number evidence established."
)


# ---------------------------------------------------------
# Glyma.02G166400
# LBD annotation independently supported at family level.
# ---------------------------------------------------------

mask = (
    lrn_evidence["Name"]
    == "Glyma.02G166400"
)

lrn_evidence.loc[
    mask,
    "annotation_consistency"
] = "supported_LBD_family"

lrn_evidence.loc[
    mask,
    "evidence_note"
] = (
    "Current annotation and independent soybean transcription-"
    "factor classification support LBD family membership. "
    "Developmentally plausible, but no exact-gene LRN evidence found."
)


# ---------------------------------------------------------
# Glyma.02G200500
# Annotation conflict across resources.
# ---------------------------------------------------------

mask = (
    lrn_evidence["Name"]
    == "Glyma.02G200500"
)

lrn_evidence.loc[
    mask,
    "annotation_consistency"
] = "discordant_across_resources"

lrn_evidence.loc[
    mask,
    "evidence_note"
] = (
    "Current gnm6 annotation is ARF18-like, whereas older "
    "resources describe an ABI3L/B3-domain transcription factor. "
    "Do not use as an ARF candidate until formal gene-model "
    "correspondence is resolved."
)


# ---------------------------------------------------------
# Glyma.02G165433
# Different ancestor naming lineage.
# ---------------------------------------------------------

mask = (
    lrn_evidence["Name"]
    == "Glyma.02G165433"
)

lrn_evidence.loc[
    mask,
    "annotation_consistency"
] = "special_cross_reference_required"

lrn_evidence.loc[
    mask,
    "evidence_note"
] = (
    "Very close to the Satt282a family anchor, but ancestorIdentifier "
    "is GmISU01.02G145800.v1.1 rather than a Wm82.a4 model. "
    "Treat historical identity cautiously."
)


display(
    lrn_evidence[
        [
            "Name",
            "start_mb",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation",
            "exact_soybean_gene_evidence",
            "soybean_development_trait_evidence",
            "annotation_consistency",
            "evidence_note"
        ]
    ]
)

### Cell 07.18 — assign evidence-aware candidate classes
* I would separate literature evidence, mechanistic plausibility, and marker proximity rather than mixing them into one score.

In [ ]:
# Cell 07.18
# Evidence-aware classification for LRN candidate genes.

def assign_lrn_evidence_class(row):

    gene = row["Name"]

    # Highest evidence:
    # exact soybean gene identified independently in a relevant
    # hormone pathway and previous developmental-trait study.
    if gene == "Glyma.02G211800":
        return "Tier_1A_independent_soybean_auxin_evidence"

    # Strong developmental mechanism + reasonably consistent annotation.
    if gene == "Glyma.02G166400":
        return "Tier_1B_developmental_regulator"

    if gene in [
        "Glyma.02G202200",
        "Glyma.02G198300",
        "Glyma.02G164148",
        "Glyma.02G163800",
    ]:
        return "Tier_2_mechanistically_plausible"

    # Marker proximity but no strong trait-specific evidence.
    if gene == "Glyma.02G165433":
        return "Tier_2_proximity_supported_uncertain_identity"

    # Annotation conflict.
    if gene == "Glyma.02G200500":
        return "Tier_3_annotation_unresolved"

    return "Tier_3_other"


lrn_evidence[
    "evidence_class"
] = (
    lrn_evidence
    .apply(
        assign_lrn_evidence_class,
        axis=1
    )
)


lrn_rank = {
    "Tier_1A_independent_soybean_auxin_evidence": 1,
    "Tier_1B_developmental_regulator": 2,
    "Tier_2_mechanistically_plausible": 3,
    "Tier_2_proximity_supported_uncertain_identity": 4,
    "Tier_3_annotation_unresolved": 5,
    "Tier_3_other": 6
}


lrn_evidence[
    "evidence_rank"
] = (
    lrn_evidence[
        "evidence_class"
    ]
    .map(lrn_rank)
)


lrn_evidence = (
    lrn_evidence
    .sort_values(
        [
            "evidence_rank",
            "distance_to_Satt282a_anchor_mb"
        ]
    )
    .reset_index(drop=True)
)


display(
    lrn_evidence[
        [
            "Name",
            "start_mb",
            "evidence_class",
            "distance_to_Satt282a_anchor_mb",
            "primary_annotation",
            "annotation_consistency"
        ]
    ]
)

### Cell 07.19 — separate biological evidence from physical proximity
* This is worth doing because otherwise Glyma.02G165433, at only ~20 kb from Satt282a, can look artificially compelling.

In [ ]:
# Cell 07.19
# Explicitly separate evidence types.
#
# Satt282a proximity is descriptive because its physical location
# comes from assay-family evidence rather than an exact SSR anchor.

lrn_evidence[
    "within_1Mb_of_Satt282a"
] = (
    lrn_evidence[
        "distance_to_Satt282a_anchor_mb"
    ] <= 1.0
)


lrn_evidence[
    "within_2Mb_of_Satt282a"
] = (
    lrn_evidence[
        "distance_to_Satt282a_anchor_mb"
    ] <= 2.0
)


print("EVIDENCE-AWARE LRN SHORTLIST")
print("=" * 90)

print(
    "Candidates:",
    len(lrn_evidence)
)

print(
    "Within 1 Mb of Satt282a:",
    int(
        lrn_evidence[
            "within_1Mb_of_Satt282a"
        ].sum()
    )
)

print(
    "Within 2 Mb of Satt282a:",
    int(
        lrn_evidence[
            "within_2Mb_of_Satt282a"
        ].sum()
    )
)


display(
    lrn_evidence[
        [
            "Name",
            "evidence_class",
            "start_mb",
            "distance_to_Satt282a_anchor_mb",
            "within_1Mb_of_Satt282a",
            "within_2Mb_of_Satt282a",
            "exact_soybean_gene_evidence"
        ]
    ]
)

### Cell 07.20 — freeze the LRN candidate-region workbook

In [ ]:
# Cell 07.20
# Final evidence-aware export for the LRN Gm02 analysis.

LRN_FINAL_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_lrn_gm02_final_candidates.xlsx"
)


lrn_region_summary = pd.DataFrame(
    [
        {
            "trait": "lrn",

            "peak_marker": "Satt282a",

            "structural_group": "pLG04",

            "physical_chr": "Gm02",

            "physical_assignment": "strong",

            "lod": 2.207883,

            "empirical_p": 0.052895,

            "significance_status":
                "suggestive_10pct",

            "region_start_bp_gnm6":
                LRN_START_BP,

            "region_end_bp_gnm6":
                LRN_END_BP,

            "region_span_mb":
                LRN_QTL["region_span_mb"],

            "region_definition":
                "anchor_defined_physical_bracket",

            "Satt282a_family_anchor_bp":
                LRN_QTL[
                    "peak_direct_anchor_bp"
                ],

            "genes_in_region":
                len(lrn_gm02_genes),

            "conservative_mechanistic_candidates":
                len(lrn_mechanistic_candidates),

            "literature_shortlist":
                len(lrn_evidence),

            "strongest_independent_evidence_gene":
                "Glyma.02G211800",

            "closest_shortlisted_gene_to_marker":
                "Glyma.02G165433",

            "important_caveat":
                (
                    "Physical bracket is not a statistical "
                    "QTL confidence interval; Satt282a coordinate "
                    "is based on assay-family evidence."
                )
        }
    ]
)


with pd.ExcelWriter(
    LRN_FINAL_FILE,
    engine="openpyxl"
) as writer:

    lrn_region_summary.to_excel(
        writer,
        sheet_name="region_summary",
        index=False
    )

    lrn_evidence.to_excel(
        writer,
        sheet_name="evidence_shortlist",
        index=False
    )

    lrn_mechanistic_candidates.to_excel(
        writer,
        sheet_name="mechanistic_13",
        index=False
    )

    lrn_near_peak_anchor.to_excel(
        writer,
        sheet_name="near_Satt282a_1Mb",
        index=False
    )

    lrn_gm02_genes.to_excel(
        writer,
        sheet_name="all_429_genes",
        index=False
    )

    annotation_reconciliation.to_excel(
        writer,
        sheet_name="annotation_conflicts",
        index=False
    )


print("FINAL LRN CANDIDATE EXPORT")
print("=" * 90)
print(LRN_FINAL_FILE)
print()
print("Notebook 07 candidate-region analysis can now be frozen.")